# QSVR M1 — Local Fidelity Kernel

This standalone notebook implements only QSVR Method 1 for shallow-water bathymetry. It uses independent Z feature-map encoding, a fidelity quantum kernel, and support vector regression with a precomputed kernel.

In [ ]:
%pip install numpy pandas matplotlib scikit-learn qiskit qiskit-aer qiskit-machine-learning

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.model_selection import StratifiedShuffleSplit
from qiskit.circuit.library import z_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
warnings.filterwarnings("ignore", category=DeprecationWarning)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load the Existing Encoded Split

This follows `VQR_Method1.ipynb` and `VQR_Method2.ipynb`: the encoded CSVs are loaded directly so every quantum model starts from the same split and `[0, π]` feature angles.

In [ ]:
train_df = pd.read_csv("train_angle_encoded.csv")
test_df = pd.read_csv("test_angle_encoded.csv")

angle_features = [
    "blue_angle",
    "green_angle",
    "log_blue_angle",
    "log_green_angle",
    "bg_ratio_angle",
    "stumpf_angle"
]
target = "depth"

X_train_full = train_df[angle_features].to_numpy(dtype=float)
y_train_full = train_df[target].to_numpy(dtype=float)
X_test_full = test_df[angle_features].to_numpy(dtype=float)
y_test_full = test_df[target].to_numpy(dtype=float)

print("Full training shape:", X_train_full.shape)
print("Full testing shape:", X_test_full.shape)
print("Train depth range:", (y_train_full.min(), y_train_full.max()))
print("Test depth range:", (y_test_full.min(), y_test_full.max()))

## 2. Runtime Configuration

Fidelity kernels require an (n \times n) training matrix. The defaults below retain observations across the 0–10 m, 10–20 m, and 20–30 m depth ranges while keeping simulator runtime manageable.

For a full experiment, set `MAX_TRAIN_SAMPLES = None` and `MAX_TEST_SAMPLES = None`.

In [ ]:
MAX_TRAIN_SAMPLES = 400
MAX_TEST_SAMPLES = 200

SVR_C = 10.0
SVR_EPSILON = 0.1

M1_REPS = 2

def depth_balanced_subset(X, y, max_samples, random_state):
    """Return a deterministic subset balanced across three depth intervals."""
    if max_samples is None or len(y) <= max_samples:
        return X.copy(), y.copy()

    depth_bins = pd.cut(
        y,
        bins=[-np.inf, 10.0, 20.0, np.inf],
        labels=False,
        include_lowest=True
    )

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        train_size=max_samples,
        random_state=random_state
    )
    selected, _ = next(splitter.split(X, depth_bins))
    return X[selected], y[selected]

X_train, y_train = depth_balanced_subset(
    X_train_full, y_train_full, MAX_TRAIN_SAMPLES, RANDOM_STATE
)
X_test, y_test = depth_balanced_subset(
    X_test_full, y_test_full, MAX_TEST_SAMPLES, RANDOM_STATE + 1
)

target_scaler = StandardScaler()
y_train_scaled = target_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

print("QSVR training shape:", X_train.shape)
print("QSVR testing shape:", X_test.shape)
print("Estimated train-kernel entries:", len(X_train) ** 2)

## 3. Shared QSVR Evaluation

All three methods produce a precomputed train kernel and a test-to-train kernel. The helper checks matrix shape and numerical validity, fits the same SVR configuration, converts predictions back to metres, and records timing plus regression metrics.

In [ ]:
results = []
predictions_by_method = {}
kernels_by_method = {}

def validate_kernel_matrices(K_train, K_test, method_name):
    K_train = np.asarray(K_train, dtype=float)
    K_test = np.asarray(K_test, dtype=float)

    expected_train = (len(X_train), len(X_train))
    expected_test = (len(X_test), len(X_train))

    if K_train.shape != expected_train:
        raise ValueError(
            f"{method_name}: expected train kernel {expected_train}, got {K_train.shape}"
        )
    if K_test.shape != expected_test:
        raise ValueError(
            f"{method_name}: expected test kernel {expected_test}, got {K_test.shape}"
        )
    if not np.isfinite(K_train).all() or not np.isfinite(K_test).all():
        raise ValueError(f"{method_name}: kernel contains NaN or infinite values")

    symmetry_error = np.max(np.abs(K_train - K_train.T))
    return K_train, K_test, symmetry_error

def fit_and_evaluate_qsvr(method_name, K_train, K_test, kernel_seconds):
    K_train, K_test, symmetry_error = validate_kernel_matrices(
        K_train, K_test, method_name
    )

    fit_start = time.perf_counter()
    model = SVR(
        kernel="precomputed",
        C=SVR_C,
        epsilon=SVR_EPSILON
    )
    model.fit(K_train, y_train_scaled)
    fit_seconds = time.perf_counter() - fit_start

    prediction_start = time.perf_counter()
    predicted_scaled = model.predict(K_test)
    prediction_seconds = time.perf_counter() - prediction_start

    predictions = target_scaler.inverse_transform(
        predicted_scaled.reshape(-1, 1)
    ).ravel()

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Method": method_name,
        "MAE (m)": mae,
        "RMSE (m)": rmse,
        "R2": r2,
        "Kernel time (s)": kernel_seconds,
        "SVR fit time (s)": fit_seconds,
        "Prediction time (s)": prediction_seconds,
        "Train symmetry error": symmetry_error
    })
    predictions_by_method[method_name] = predictions
    kernels_by_method[method_name] = (K_train, K_test)

    print("=" * 62)
    print(f"{method_name} RESULTS")
    print("=" * 62)
    print(f"MAE                 : {mae:.4f} m")
    print(f"RMSE                : {rmse:.4f} m")
    print(f"R2                  : {r2:.4f}")
    print(f"Kernel build time   : {kernel_seconds:.2f} s")
    print(f"SVR fit time        : {fit_seconds:.2f} s")
    print(f"Prediction time     : {prediction_seconds:.4f} s")
    print(f"Kernel symmetry err.: {symmetry_error:.3e}")

    return model, predictions

## 4. QSVR M1 — Local Fidelity Quantum Kernel

M1 uses independent single-qubit phase encoding through `ZFeatureMap`. It is the least entangled QSVR baseline and isolates the contribution of local quantum feature encoding.

In [ ]:
n_qubits = len(angle_features)

m1_feature_map = z_feature_map(
    feature_dimension=n_qubits,
    reps=M1_REPS
)
m1_quantum_kernel = FidelityQuantumKernel(
    feature_map=m1_feature_map
)

print("QSVR M1 feature map")
print(m1_feature_map.decompose().draw(output="text"))

In [ ]:
m1_start = time.perf_counter()

K_train_m1 = m1_quantum_kernel.evaluate(x_vec=X_train)
K_test_m1 = m1_quantum_kernel.evaluate(
    x_vec=X_test,
    y_vec=X_train
)

m1_kernel_seconds = time.perf_counter() - m1_start

m1_model, m1_predictions = fit_and_evaluate_qsvr(
    "QSVR M1 - Local Fidelity",
    K_train_m1,
    K_test_m1,
    m1_kernel_seconds
)

## Result Visualization

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, m1_predictions, alpha=0.5, color="#D97745")
plt.plot([0, 30], [0, 30], "k--", label="Ideal prediction")
plt.xlabel("Actual depth (m)")
plt.ylabel("Predicted depth (m)")
plt.title("QSVR M1 — Local Fidelity Kernel\\nR2 = " + f"{r2_score(y_test, m1_predictions):.3f}")
plt.xlim(0, 30)
plt.ylim(0, 30)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Interpretation

M1 is the non-entangled quantum-kernel baseline. Its full fidelity matrix scales quadratically with the sample count.

For a final scientific comparison, run the same observations across all methods and replace the random split with spatial validation.